# Transformer Tanks

A distribution transformer bank is one `PowerTransformer` made of one or more physical tanks. Describing
each tank's lugs, core and windings captures connections that ends alone cannot, such as a center-tapped
secondary or an open-wye open-delta bank. There are two ways to build one:

- `TransformerBankFromConfigurationBuilder`, from a `TransformerTankConfiguration`: the tank's design,
  built once and shared by every transformer of that type,
- `TransformerBankBuilder`, with each tank's lugs, core and windings built on the transformer itself.

```mermaid
flowchart LR
    config["TransformerTankConfiguration<br/>lugs, core, windings,<br/>mesh impedance, losses"]
    config --- tank1["TransformerTank<br/>xf_2341_tank"] --- xf1["PowerTransformer xf_2341<br/>ends and phase ends"]
    config --- tank2["TransformerTank<br/>xf_2342_tank"] --- xf2["PowerTransformer xf_2342<br/>ends and phase ends"]
```

## Set up

The CIM profile and unit handling are process-wide environment variables, so set them before importing
anything from CIMantic Graphs. `new_feeder_model` makes an empty model to build into.

In [1]:
import os

os.environ["CIMG_CIM_PROFILE"] = "cim18v17"
os.environ["CIMG_USE_UNITS"] = "TRUE"
os.environ["CIMG_SYSTEM_MVA_BASE"] = "100"

from cimtables.units import Qty

from cimbuilder.network import new_feeder_model
from cimbuilder.object_builder import (
    BaseVoltageBuilder,
    TransformerBankBuilder,
    TransformerBankFromConfigurationBuilder,
    TransformerTankConfigurationBuilder,
)

network = new_feeder_model(
    authority="https://example.com/models",
    model="transformer_tanks",
    Feeder="fdr_demo",
)
container = network.container
base_voltage_builder = BaseVoltageBuilder(network, container)
for mRID, nominalVoltage in [
    ("bv_12_47kV", Qty(12.47, "kV")),
    ("bv_240V", Qty(240, "V")),
]:
    base_voltage_builder.create(mRID=mRID, name=mRID, nominalVoltage=nominalVoltage)

## A tank configuration

This is a 25 kVA single-phase pole-top transformer with a center-tapped secondary: a 7.2 kV primary
winding between lugs H1 and H2, and two 120 V secondary half-windings, X1-X2 and X2-X3, with X2 the
center tap.

- `add_electrical` sets the losses from the test report.
- Each `add_transformer_lug` adds one lug, named `<configuration>_lug_<designation>`. Its
  `sequenceNumber` is the conductor it lands on, matched later against each transformer's phase ends.
- `add_transformer_core` adds the core, and `add_transformer_winding` a winding between two lugs.
- `add_mesh_impedance` sets the impedance between two windings. Percent values are on the base of the
  From winding.

In [2]:
config_builder = TransformerTankConfigurationBuilder(network, container)
config = config_builder.create(mRID="xc_ct_25kVA", name="25 kVA center-tap")
config_builder.add_electrical(
    config,
    loadLoss=Qty(300, "W"),
    noLoadLoss=Qty(100, "W"),
    percentImpedance=1.8,
    percentMagnetizingCurrent=0.5,
)
for designation, sequenceNumber in [
    ("H1", 1),
    ("H2", 2),
    ("X1", 3),
    ("X3", 4),
    ("X2", 5),
]:
    config_builder.add_transformer_lug(
        config, designation=designation, sequenceNumber=sequenceNumber
    )
config, core = config_builder.add_transformer_core(config, form="shellForm")

windings = {}
for from_lug, to_lug, ratedU, ratedS in [
    ("h1", "h2", Qty(7.2, "kV"), Qty(25, "kVA")),
    ("x1", "x2", Qty(120, "V"), Qty(12.5, "kVA")),
    ("x2", "x3", Qty(120, "V"), Qty(12.5, "kVA")),
]:
    config, windings[from_lug + to_lug] = config_builder.add_transformer_winding(
        config,
        FromTransformerLug=f"xc_ct_25kVA_lug_{from_lug}",
        ToTransformerLug=f"xc_ct_25kVA_lug_{to_lug}",
        TransformerCore=core.mRID,
        connection="I",
        ratedU=ratedU,
        baseU=ratedU,
        ratedS=ratedS,
        baseS=Qty(25, "kVA"),
    )
for half in ("x1x2", "x2x3"):
    config_builder.add_mesh_impedance(
        config,
        FromWinding=windings["h1h2"].mRID,
        ToWinding=windings[half].mRID,
        r=Qty(1.2, "percent"),
        x=Qty(2.0, "percent"),
    )

print([winding.mRID for winding in windings.values()])
mesh = windings["h1h2"].FromMeshImpedance[0]
print(f"{mesh.mRID}: r = {mesh.r.to('ohm'):.2f} ohm, referred to the primary")

['xc_ct_25kVA_wdg_h1h2', 'xc_ct_25kVA_wdg_x1x2', 'xc_ct_25kVA_wdg_x2x3']
xc_ct_25kVA_mesh_h1h2x1x2: r = 24.88 ohm, referred to the primary


## Transformers built from the configuration

Each transformer then needs only what is particular to it:

- its ends, with the nodes they connect to and the phases of their conductors in order,
- a tank that refers to the configuration,
- a `TransformerPhaseEnd` for each conductor, giving the phase that lands on each lug `sequenceNumber`.

Here two transformers of the same type serve different phases.

In [3]:
bank_builder = TransformerBankFromConfigurationBuilder(network, container)


def pole_top(mRID, phase):
    xfmr = bank_builder.create(mRID=mRID)
    bank_builder.add_connectivity(xfmr)
    bank_builder.add_power_transformer_end(
        xfmr,
        node=f"{mRID}_hv",
        endNumber=1,
        phases=f"{phase}N",
        connectionKind="I",
        BaseVoltage="bv_12_47kV",
    )
    bank_builder.add_power_transformer_end(
        xfmr,
        node=f"{mRID}_lv",
        endNumber=2,
        phases="s1s2N",
        connectionKind="I",
        BaseVoltage="bv_240V",
    )
    bank_builder.add_transformer_tank(xfmr, TankConfiguration="xc_ct_25kVA")
    for endNumber, sequenceNumber, conductor, phaseNeutralU in [
        (1, 1, phase, Qty(7.2, "kV")),
        (1, 2, "N", 0),
        (2, 3, "s1", Qty(120, "V")),
        (2, 4, "s2", Qty(120, "V")),
        (2, 5, "N", 0),
    ]:
        bank_builder.add_transformer_phase_end(
            xfmr,
            endNumber=endNumber,
            sequenceNumber=sequenceNumber,
            phase=conductor,
            phaseNeutralU=phaseNeutralU,
        )
    return xfmr


for xfmr in (pole_top("xf_2341", "A"), pole_top("xf_2342", "C")):
    [tank] = xfmr.TransformerTanks
    terminals = sorted(xfmr.Terminals, key=lambda terminal: terminal.sequenceNumber)
    print(
        xfmr.mRID,
        tank.mRID,
        "->",
        tank.TankConfiguration.mRID,
        [terminal.phases.value for terminal in terminals],
    )

xf_2341 xf_2341_tank -> xc_ct_25kVA ['AN', 's12N']
xf_2342 xf_2342_tank -> xc_ct_25kVA ['CN', 's12N']


## A bank with its tanks built directly

`TransformerBankBuilder` builds the same pieces, lugs, core, windings and mesh impedance, on a tank of the
transformer itself, named after the tank (`<tank>_lug_<designation>`, `<tank>_wdg_<lugs>`). Use it for a
one-off bank, or a bank of tanks that differ from each other, such as the lighting and power tanks of an
open-wye open-delta bank.

The lugs of a directly built tank refer to the transformer's own phase ends, so add those first.

In [4]:
direct_builder = TransformerBankBuilder(network, container)
xfmr = direct_builder.create(mRID="xf_2400")
direct_builder.add_connectivity(xfmr)
direct_builder.add_power_transformer_end(
    xfmr,
    node="xf_2400_hv",
    endNumber=1,
    phases="BN",
    connectionKind="I",
    BaseVoltage="bv_12_47kV",
)
direct_builder.add_power_transformer_end(
    xfmr,
    node="xf_2400_lv",
    endNumber=2,
    phases="s1s2N",
    connectionKind="I",
    BaseVoltage="bv_240V",
)
for endNumber, sequenceNumber, conductor in [
    (1, 1, "B"),
    (1, 2, "N"),
    (2, 3, "s1"),
    (2, 4, "s2"),
    (2, 5, "N"),
]:
    direct_builder.add_transformer_phase_end(
        xfmr, endNumber=endNumber, sequenceNumber=sequenceNumber, phase=conductor
    )

xfmr, tank = direct_builder.add_transformer_tank(
    xfmr, loadLoss=Qty(300, "W"), noLoadLoss=Qty(100, "W"), percentImpedance=1.8
)
for designation, sequenceNumber, phase_end in [
    ("H1", 1, "xf_2400_1_b"),
    ("H2", 2, "xf_2400_1_n"),
    ("X1", 3, "xf_2400_2_s1"),
    ("X3", 4, "xf_2400_2_s2"),
    ("X2", 5, "xf_2400_2_n"),
]:
    direct_builder.add_transformer_lug(
        xfmr,
        TransformerTank=tank.mRID,
        designation=designation,
        sequenceNumber=sequenceNumber,
        TransformerEnd=phase_end,
    )
xfmr, core = direct_builder.add_transformer_core(
    xfmr, TransformerTank=tank.mRID, form="shellForm"
)
for from_lug, to_lug, ratedU in [
    ("h1", "h2", Qty(7.2, "kV")),
    ("x1", "x2", Qty(120, "V")),
    ("x2", "x3", Qty(120, "V")),
]:
    direct_builder.add_transformer_winding(
        xfmr,
        TransformerTank=tank.mRID,
        FromTransformerLug=f"{tank.mRID}_lug_{from_lug}",
        ToTransformerLug=f"{tank.mRID}_lug_{to_lug}",
        TransformerCore=core.mRID,
        connection="I",
        ratedU=ratedU,
        baseU=ratedU,
        baseS=Qty(25, "kVA"),
    )
for half in ("x1x2", "x2x3"):
    direct_builder.add_mesh_impedance(
        xfmr,
        FromWinding=f"{tank.mRID}_wdg_h1h2",
        ToWinding=f"{tank.mRID}_wdg_{half}",
        r=Qty(1.2, "percent"),
        x=Qty(2.0, "percent"),
    )

print(tank.mRID)
for lug in sorted(tank.TransformerLug, key=lambda lug: lug.sequenceNumber):
    print(
        f"  {lug.mRID} on conductor {lug.sequenceNumber:g}, {lug.TransformerEnd.mRID}"
    )

xf_2400_tank
  xf_2400_tank_lug_h1 on conductor 1, xf_2400_1_b
  xf_2400_tank_lug_h2 on conductor 2, xf_2400_1_n
  xf_2400_tank_lug_x1 on conductor 3, xf_2400_2_s1
  xf_2400_tank_lug_x3 on conductor 4, xf_2400_2_s2
  xf_2400_tank_lug_x2 on conductor 5, xf_2400_2_n
